In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(".")

FILES = {
    "mixit_model_a": BASE_DIR / "scenario-mixit_only-model-a.csv",
    "mixit_model_b": BASE_DIR / "scenario-mixit_only-model-b.csv",
    "ori_model_a": BASE_DIR / "scenario-ori_only-model-a.csv",
    "ori_model_b": BASE_DIR / "scenario-ori_only-model-b.csv",
    "overall_model_a": BASE_DIR / "scenario-overall-model-a.csv",
    "overall_model_b": BASE_DIR / "scenario-overall-model-b.csv",
    "label_ori": BASE_DIR / "test-label-ori.xlsx",
    "label_a": BASE_DIR / "test-label-a.xlsx",
    "label_mixit": BASE_DIR / "test-label-mixit.xlsx",
}

data = {}

for name, path in FILES.items():
    try:
        if path.suffix == ".csv":
            df = pd.read_csv(
                path,
                engine="python",      # FIX: tolerant parser
                sep=None,             # auto-detect delimiter
                on_bad_lines="skip"   # skip broken rows safely
            )
        else:
            df = pd.read_excel(path)

        data[name] = df

        print(f"\n=== {name} ===")
        print("Shape:", df.shape)
        print("Columns:")
        for col in df.columns:
            print(" -", col)

    except Exception as e:
        print(f"\n!!! Failed to load {name}")
        print(e)

In [ ]:
data["mixit_model_a"]
data["label_ori"]

In [ ]:
def show_head(df, name, n=10):
    print(f"\n--- {name} (first {n} rows) ---")
    print(df.head(n))
    print("Shape:", df.shape)


# 1️⃣ Raw prediction files
show_head(data["mixit_model_a"], "mixit_model_a RAW")
show_head(data["mixit_model_b"], "mixit_model_b RAW")

show_head(data["ori_model_a"], "ori_model_a RAW")
show_head(data["ori_model_b"], "ori_model_b RAW")

show_head(data["overall_model_a"], "overall_model_a RAW")
show_head(data["overall_model_b"], "overall_model_b RAW")


# 2️⃣ Raw label files
show_head(data["label_mixit"], "label_mixit RAW")
show_head(data["label_ori"], "label_ori RAW")
show_head(data["label_a"], "label_a RAW")


# 3️⃣ After preparation
mixit_a_prep = prepare_predictions(data["mixit_model_a"])
mixit_b_prep = prepare_predictions(data["mixit_model_b"])

show_head(mixit_a_prep, "mixit_model_a PREPARED")
show_head(mixit_b_prep, "mixit_model_b PREPARED")

label_mixit_prep = prepare_labels(data["label_mixit"])
show_head(label_mixit_prep, "label_mixit PREPARED")


# 4️⃣ After merge (this is the critical one)
mixit_merge_a = mixit_a_prep.merge(label_mixit_prep, on="File", how="inner")
mixit_merge_b = mixit_b_prep.merge(label_mixit_prep, on="File", how="inner")

show_head(mixit_merge_a, "mixit A MERGED")
show_head(mixit_merge_b, "mixit B MERGED")


# 5️⃣ Show unique examples to spot mismatch
print("\n--- Example prediction filenames ---")
print(mixit_a_prep["File"].astype(str).head(10).tolist())

print("\n--- Example label filenames ---")
print(label_mixit_prep["File"].astype(str).head(10).tolist())


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from statsmodels.stats.contingency_tables import mcnemar

BASE_DIR = Path(".")

FILES = {
    "mixit_model_a": BASE_DIR / "scenario-mixit_only-model-a.csv",
    "mixit_model_b": BASE_DIR / "scenario-mixit_only-model-b.csv",
    "ori_model_a": BASE_DIR / "scenario-ori_only-model-a.csv",
    "ori_model_b": BASE_DIR / "scenario-ori_only-model-b.csv",
    "overall_model_a": BASE_DIR / "scenario-overall-model-a.csv",
    "overall_model_b": BASE_DIR / "scenario-overall-model-b.csv",
    "label_ori": BASE_DIR / "test-label-ori.xlsx",
    "label_a": BASE_DIR / "test-label-a.xlsx",
    "label_mixit": BASE_DIR / "test-label-mixit.xlsx",
}

data = {}
for name, path in FILES.items():
    if path.suffix == ".csv":
        df = pd.read_csv(path, engine="python", sep=None, on_bad_lines="skip")
    else:
        df = pd.read_excel(path)
    data[name] = df

def normalize_filename(df, col="File"):
    df = df.copy()
    df[col] = df[col].astype(str).apply(lambda x: Path(x).name)
    return df

def prepare_predictions(df_pred):
    df_pred = df_pred.copy()
    df_pred = normalize_filename(df_pred, "File")
    
    results = []
    
    for file in df_pred["File"].unique():
        file_preds = df_pred[df_pred["File"] == file].copy()
        
        vote_counts = file_preds["Scientific name"].value_counts()
        max_votes = vote_counts.max()
        
        tied_labels = vote_counts[vote_counts == max_votes].index.tolist()
        
        if len(tied_labels) == 1:
            final_label = tied_labels[0]
        else:
            tied_preds = file_preds[file_preds["Scientific name"].isin(tied_labels)]
            final_label = tied_preds.loc[tied_preds["Confidence"].idxmax(), "Scientific name"]
        
        results.append({
            "File": file,
            "Scientific name": final_label
        })
    
    return pd.DataFrame(results)

def prepare_labels(df_label):
    df = df_label.rename(columns={"nama_file": "File", "spesies": "TrueLabel"})
    df = normalize_filename(df, "File")
    return df

def build_correctness(df_pred, df_label):
    df = df_pred.merge(df_label, on="File", how="inner")
    df["Correct"] = df["Scientific name"] == df["TrueLabel"]
    return df[["File", "Correct"]]

def compute_mcnemar(correct_a, correct_b):
    df = correct_a.merge(correct_b, on="File", suffixes=("_A", "_B"))
    
    a = ((df["Correct_A"] == True) & (df["Correct_B"] == True)).sum()
    b = ((df["Correct_A"] == True) & (df["Correct_B"] == False)).sum()
    c = ((df["Correct_A"] == False) & (df["Correct_B"] == True)).sum()
    d = ((df["Correct_A"] == False) & (df["Correct_B"] == False)).sum()
    
    table = [[a, b], [c, d]]
    result = mcnemar(table, exact=False, correction=True)
    
    return {
        "both_correct": a,
        "a_correct_b_wrong": b,
        "a_wrong_b_correct": c,
        "both_wrong": d,
        "statistic": result.statistic,
        "pvalue": result.pvalue,
        "table": table
    }

label_ori = prepare_labels(data["label_ori"])
label_a = prepare_labels(data["label_a"])
label_mixit = prepare_labels(data["label_mixit"])

print("Processing predictions with majority voting + confidence tie-breaking...")

mixit_a = prepare_predictions(data["mixit_model_a"])
mixit_b = prepare_predictions(data["mixit_model_b"])
ori_a = prepare_predictions(data["ori_model_a"])
ori_b = prepare_predictions(data["ori_model_b"])
overall_a = prepare_predictions(data["overall_model_a"])
overall_b = prepare_predictions(data["overall_model_b"])

print(f"MIXIT: Model A -> {len(mixit_a)} files, Model B -> {len(mixit_b)} files")
print(f"ORI: Model A -> {len(ori_a)} files, Model B -> {len(ori_b)} files")
print(f"OVERALL: Model A -> {len(overall_a)} files, Model B -> {len(overall_b)} files")

mixit_corr_a = build_correctness(mixit_a, label_mixit)
mixit_corr_b = build_correctness(mixit_b, label_mixit)
ori_corr_a = build_correctness(ori_a, label_ori)
ori_corr_b = build_correctness(ori_b, label_ori)
overall_corr_a = build_correctness(overall_a, label_a)
overall_corr_b = build_correctness(overall_b, label_a)

results = {
    "MIXIT": compute_mcnemar(mixit_corr_a, mixit_corr_b),
    "ORI": compute_mcnemar(ori_corr_a, ori_corr_b),
    "OVERALL": compute_mcnemar(overall_corr_a, overall_corr_b)
}

fig = plt.figure(figsize=(16, 10))

for idx, (scenario, res) in enumerate(results.items()):
    ax = plt.subplot(2, 3, idx + 1)
    
    table_data = np.array(res["table"])
    
    sns.heatmap(table_data, annot=True, fmt='d', cmap='Blues', 
                cbar_kws={'label': 'Count'}, 
                xticklabels=['Model B Correct', 'Model B Wrong'],
                yticklabels=['Model A Correct', 'Model A Wrong'],
                ax=ax, square=True, linewidths=2, linecolor='black')
    
    ax.set_title(f'{scenario}\nχ²={res["statistic"]:.3f}, p={res["pvalue"]:.4f}', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Model B', fontsize=10)
    ax.set_ylabel('Model A', fontsize=10)

for idx, (scenario, res) in enumerate(results.items()):
    ax = plt.subplot(2, 3, idx + 4)
    
    categories = ['Both\nCorrect', 'A Correct\nB Wrong', 'A Wrong\nB Correct', 'Both\nWrong']
    values = [res["both_correct"], res["a_correct_b_wrong"], 
              res["a_wrong_b_correct"], res["both_wrong"]]
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#95a5a6']
    
    bars = ax.bar(categories, values, color=colors, edgecolor='black', linewidth=1.5)
    
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val}\n({val/sum(values)*100:.1f}%)',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax.set_title(f'{scenario} - Agreement Distribution', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Samples', fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_ylim(0, max(values) * 1.15)

plt.tight_layout()
plt.savefig('mcnemar_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== McNemar Test Summary ===")
for scenario, res in results.items():
    total = sum([res["both_correct"], res["a_correct_b_wrong"], 
                 res["a_wrong_b_correct"], res["both_wrong"]])
    acc_a = (res["both_correct"] + res["a_correct_b_wrong"]) / total * 100
    acc_b = (res["both_correct"] + res["a_wrong_b_correct"]) / total * 100
    
    print(f"\n{scenario}:")
    print(f"  Total samples: {total}")
    print(f"  Model A accuracy: {acc_a:.2f}%")
    print(f"  Model B accuracy: {acc_b:.2f}%")
    print(f"  Both correct: {res['both_correct']} ({res['both_correct']/total*100:.1f}%)")
    print(f"  A correct, B wrong: {res['a_correct_b_wrong']} ({res['a_correct_b_wrong']/total*100:.1f}%)")
    print(f"  A wrong, B correct: {res['a_wrong_b_correct']} ({res['a_wrong_b_correct']/total*100:.1f}%)")
    print(f"  Both wrong: {res['both_wrong']} ({res['both_wrong']/total*100:.1f}%)")
    print(f"  χ² statistic: {res['statistic']:.4f}")
    print(f"  p-value: {res['pvalue']:.4f}")
    
    if res['pvalue'] < 0.05:
        if res['a_correct_b_wrong'] > res['a_wrong_b_correct']:
            print(f"  → Model A significantly better than Model B")
        else:
            print(f"  → Model B significantly better than Model A")
    else:
        print(f"  → No significant difference between models")